
# Image kernels: filter, find, and measure

``tttrlib`` carries a family of 2-D image kernels -- median / min / max /
midpoint filters, fast gaussian blur, Sobel and Laplace-8 derivatives, area /
bilinear / bicubic resampling, affine warping, summed-area tables, value
histograms, intensity moments, and frame-drift estimation. They are plain
header-only C++ under the hood (concepts re-expressed from the MIT-licensed
[Simd library](https://github.com/ermig1979/Simd)_), exposed here as NumPy
in / NumPy out calls.

This example walks the workflow a photon-counting image goes through:

1. **denoise** -- a median filter removes impulse noise without blurring
   edges; the gaussian blur is for smoother backgrounds;
2. **localise** -- the intensity moments give a spot's centroid and width
   without a fit;
3. **register** -- two frames of a drifting sample are aligned by the pyramid
   drift estimator, sub-pixel;
4. **measure** -- a summed-area table answers arbitrary rectangle sums in
   constant time, and the histogram shows the value distribution.


In [ ]:
import numpy as np

import tttrlib

rng = np.random.RandomState(3)

# ----------------------------------------------------------------------------
# A synthetic photon-counting frame: two gaussian spots on a background,
# impulse noise (hot pixels), and readout gradient.
# ----------------------------------------------------------------------------
yy, xx = np.mgrid[0:96, 0:96]
spots = (
    400.0 * np.exp(-((xx - 41.3) ** 2 + (yy - 55.7) ** 2) / (2 * 2.8**2))
    + 250.0 * np.exp(-((xx - 62.8) ** 2 + (yy - 28.2) ** 2) / (2 * 3.5**2))
    + 20.0
)
counts = rng.poisson(spots).astype(np.uint16)
hot = rng.rand(96, 96) < 0.01  # 1% hot pixels
counts[hot] = (counts[hot].astype(np.int64) + rng.randint(800, 1200, int(hot.sum()))).astype(np.uint16)

# Step 1: median denoise. On integer pixels the sliding-histogram median is
# exact -- the histogram *is* the window multiset -- and drops the hot pixels
# without touching spot edges.
clean = tttrlib.median_filter(counts, "square3x3")
removed = int((counts[hot] > clean[hot]).sum())
print(f"median filter removed the impulse in {removed} of {hot.sum()} hot pixels")

# The fast gaussian (three box blurs, cost independent of sigma) for a smooth
# background estimate -- a display/analysis kernel, not scipy-exact.
background = tttrlib.gaussian_blur(clean.astype(np.float64), 8.0)

# Step 2: intensity moments locate a spot. Mask to the left spot's quadrant so
# the second spot does not pull the centroid.
mask = np.zeros((96, 96), dtype=bool)
mask[45:68, 28:55] = True
m = tttrlib.moments((clean - background).clip(min=0), mask)
cx, cy = m[1] / m[0], m[2] / m[0]
print(f"moment centroid of the left spot: ({cx:.2f}, {cy:.2f}) -- true (41.3, 55.7)")

# Step 3: drift. The second frame is the same scene translated by (2.6, -1.4)
# px (bilinear), plus independent noise. One call recovers the translation.
shifted = np.roll(np.roll(clean.astype(np.float64), 3, axis=1), -1, axis=0)
shifted += rng.normal(0, 2.0, shifted.shape)
dx, dy, score = tttrlib.estimate_drift(clean.astype(np.float64), np.ascontiguousarray(shifted), 8)
print(f"drift estimate: (dx, dy) = ({dx:.2f}, {dy:.2f}), SAD score {score:.3f}")

# Step 4: summed-area table -- one pass, then every rectangle sum is four
# lookups. Integrate the denoised counts and read a region around the spot.
table = tttrlib.integral_image_u16(clean)
s = tttrlib.rect_sum_u64(table, 96, 50, 36, 62, 48)
print(f"rectangle sum over the spot box: {s:.0f} counts "
      f"(direct: {clean[50:63, 36:49].sum()})")

# The value distribution before and after denoising: the hot-pixel tail is
# gone from the histogram of the median-filtered frame.
hist_raw = tttrlib.image_histogram(clean, 0, 1200)
print(f"bins above 800 counts: raw {int((hist_raw[800:] > 0).sum())} "
      f"(the median removed the impulse tail)")

# Derivatives and resampling for display: the Sobel magnitude highlights the
# spot edges; the area resize is the anti-aliased way to shrink a frame.
gx, gy = tttrlib.sobel_dx(clean.astype(np.float64)), tttrlib.sobel_dy(clean.astype(np.float64))
magnitude = np.hypot(gx, gy)
small = tttrlib.resize(clean, 48, 48, "area")
print(f"Sobel magnitude peak {magnitude.max():.1f} at the spot rim; "
      f"area-resized frame {small.shape} for a quick look")